# Data Engineering — Moduł 2: Apache Airflow — podstawy

**Apache Airflow** to najpopularniejsze narzędzie do **orkiestracji** potoków danych --
definiujesz w Pythonie, JAKIE kroki mają się wykonać, W JAKIEJ KOLEJNOŚCI i JAK CZĘSTO, a
Airflow pilnuje harmonogramu, kolejności, ponawiania błędów i historii uruchomień. W tym
module piszemy i uruchamiamy pierwsze DAG-i.

## Spis treści
1. [Architektura Airflow](#sec1)
2. [DAG i pierwszy task](#sec2)
3. [Zależności między taskami](#sec3)
4. [BashOperator i mieszanie operatorów](#sec4)
5. [XComs — przekazywanie danych między taskami](#sec5)
6. [Harmonogramy: cron, presety, catchup](#sec6)
7. [Podsumowanie i ćwiczenia](#sec7)


Uruchamiamy Airflow lokalnie -- w prawdziwym wdrożeniu `AIRFLOW_HOME` i baza metadanych (tu: SQLite, produkcyjnie zwykle PostgreSQL) są konfigurowane raz dla całego zespołu, a nie w każdym notebooku osobno. Robimy to tutaj, żeby każdy moduł kursu działał niezależnie od pozostałych.

In [ ]:
import os
import subprocess

# Kazdy modul dotyczacy Airflow uzywa wlasnego, izolowanego AIRFLOW_HOME (katalogu
# roboczego z metadanymi, DAG-ami i logami) -- tak, jak w prawdziwym wdrozeniu, gdzie
# AIRFLOW_HOME jest ustawiane raz dla calego srodowiska.
AIRFLOW_HOME = os.path.abspath("airflow_home")
AIRFLOW_BIN = "/tmp/airflow_venv/bin/airflow"
os.makedirs(f"{AIRFLOW_HOME}/dags", exist_ok=True)

env = os.environ.copy()
env["AIRFLOW_HOME"] = AIRFLOW_HOME
env["AIRFLOW__CORE__LOAD_EXAMPLES"] = "False"
env["AIRFLOW__LOGGING__LOGGING_LEVEL"] = "WARNING"


def airflow(*args, check=True):
    """Uruchamia komende `airflow ...` w izolowanym AIRFLOW_HOME i wypisuje wynik."""
    wynik = subprocess.run([AIRFLOW_BIN, *args], env=env, capture_output=True, text=True)
    print(wynik.stdout)
    if wynik.returncode != 0:
        print(wynik.stderr[-3000:])
        if check:
            raise RuntimeError(f"airflow {' '.join(args)} zakonczyl sie kodem {wynik.returncode}")
    return wynik


# Inicjalizacja bazy metadanych (SQLite) -- potrzebna raz na AIRFLOW_HOME
airflow("db", "migrate")


<a id="sec1"></a>
## 1. Architektura Airflow

Airflow składa się z kilku współpracujących komponentów:

- **Scheduler** -- proces w tle, który non-stop sprawdza harmonogramy DAG-ów i decyduje,
  KIEDY uruchomić kolejne zadania.
- **Webserver** -- interfejs graficzny (przeglądarka), w którym widzisz DAG-i, historię
  uruchomień, logi zadań -- w tym kursie NIE uruchamiamy webservera (wymaga stale
  działającego procesu), tylko pracujemy z poziomu CLI (`airflow dags test`, dokładnie
  tak, jak zrobiliśmy to w konfiguracji powyżej).
- **Executor** -- decyduje, JAK fizycznie uruchamiane są zadania (sekwencyjnie na jednej
  maszynie -- `SequentialExecutor`/`LocalExecutor`, domyślne lokalnie, albo rozproszone
  na wielu workerach -- `CeleryExecutor`/`KubernetesExecutor` w produkcji).
- **Metadata Database** -- baza (u nas SQLite, w produkcji zwykle PostgreSQL) z historią
  wszystkich uruchomień, statusami zadań i konfiguracją.
- **DAG-i** -- pliki Pythona w folderze `dags/`, które OPISUJĄ potoki (nie wykonują ich
  bezpośrednio -- to rola schedulera/executora).

> 🏭 **`airflow dags test` a prawdziwy harmonogram**
>
> Komenda `airflow dags test DAG_ID DATA`, której używamy w tym kursie, uruchamia JEDNO wykonanie DAG-a natychmiast, z pominięciem schedulera -- świetne do nauki i debugowania. W prawdziwym wdrożeniu scheduler działa CIĄGLE w tle i sam decyduje, kiedy uruchomić DAG zgodnie z jego harmonogramem (sekcja 6).

<a id="sec2"></a>
## 2. DAG i pierwszy task

**DAG** (*Directed Acyclic Graph* -- skierowany graf acykliczny) to po prostu nazwa dla
"potoku zadań z zależnościami, bez cykli". W Airflow DAG to obiekt Pythona, a **task**
(zadanie) to pojedynczy krok wewnątrz niego, zbudowany przez **Operator** -- gotowy
"szablon" konkretnego typu pracy. `PythonOperator` uruchamia dowolną funkcję Pythona.

In [ ]:
dag_kod = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime


def przywitaj():
    print("Czesc z pierwszego DAG-a!")


with DAG(
    dag_id="m2_pierwszy_dag",
    start_date=datetime(2024, 1, 1),
    schedule=None,          # brak automatycznego harmonogramu -- uruchamiamy recznie
    catchup=False,
) as dag:
    powitanie = PythonOperator(task_id="powitanie", python_callable=przywitaj)
'''

with open(f"{AIRFLOW_HOME}/dags/m2_pierwszy_dag.py", "w") as f:
    f.write(dag_kod)

print("Zapisano DAG do folderu dags/.")


In [ ]:
airflow("dags", "test", "m2_pierwszy_dag", "2024-01-01")


> ⚡ **`with DAG(...) as dag:` to context manager**
>
> Każdy Operator utworzony WEWNĄTRZ bloku `with DAG(...) as dag:` automatycznie "dołącza się" do tego DAG-a -- nie musisz przekazywać `dag=dag` do każdego Operatora osobno (choć starszy styl Airflow 1.x tego wymagał).

<a id="sec3"></a>
## 3. Zależności między taskami

Prawdziwa moc Airflow ujawnia się, gdy DAG ma WIELE zadań w określonej kolejności.
Zależności zapisujemy operatorem `>>` ("potem") albo `<<` ("przedtem") -- czytelny,
graficzny zapis grafu zależności wprost w kodzie.

In [ ]:
dag_kod = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime


def wczytaj_dane():
    print("Krok 1: wczytuje surowe dane...")


def wyczysc_dane():
    print("Krok 2: czyszcze dane (duplikaty, braki)...")


def policz_agregaty():
    print("Krok 3a: licze agregaty sprzedazy...")


def zbuduj_raport_klientow():
    print("Krok 3b: buduje raport segmentow klientow...")


def wyslij_powiadomienie():
    print("Krok 4: wysylam powiadomienie, ze dane sa gotowe.")


with DAG(
    dag_id="m2_potok_z_zaleznosciami",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    t_wczytaj = PythonOperator(task_id="wczytaj_dane", python_callable=wczytaj_dane)
    t_czysc = PythonOperator(task_id="wyczysc_dane", python_callable=wyczysc_dane)
    t_agregaty = PythonOperator(task_id="policz_agregaty", python_callable=policz_agregaty)
    t_raport = PythonOperator(task_id="zbuduj_raport_klientow", python_callable=zbuduj_raport_klientow)
    t_powiadom = PythonOperator(task_id="wyslij_powiadomienie", python_callable=wyslij_powiadomienie)

    # wczytaj -> czysc -> (agregaty ORAZ raport ROWNOLEGLE) -> powiadom
    t_wczytaj >> t_czysc >> [t_agregaty, t_raport] >> t_powiadom
'''

with open(f"{AIRFLOW_HOME}/dags/m2_potok_z_zaleznosciami.py", "w") as f:
    f.write(dag_kod)

airflow("dags", "test", "m2_potok_z_zaleznosciami", "2024-01-01")


> ⚡ **Lista w `>>` oznacza RÓWNOLEGŁOŚĆ**
>
> `t_czysc >> [t_agregaty, t_raport]` mówi: oba zadania w liście mogą wystartować RÓWNOCZEŚNIE, jak tylko `t_czysc` się zakończy -- nie czekają na siebie nawzajem. To dokładnie ten sam pomysł co równoległe taski w Module 6 kursu Databricks (Workflows -- multi-task jobs).

<a id="sec4"></a>
## 4. BashOperator i mieszanie operatorów

`BashOperator` uruchamia dowolną komendę powłoki -- niezastąpiony, gdy krok potoku to
skrypt, narzędzie CLI (jak `dbt run`, które poznamy w Module 4) albo dowolny program spoza
świata Pythona. DAG może swobodnie MIESZAĆ różne typy operatorów.

In [ ]:
dag_kod = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from datetime import datetime


def przygotuj_dane():
    print("Przygotowuje plik z danymi...")


with DAG(
    dag_id="m2_mieszane_operatory",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    t_python = PythonOperator(task_id="przygotuj_dane", python_callable=przygotuj_dane)
    t_bash = BashOperator(task_id="pokaz_date", bash_command='echo "Zadanie uruchomione: $(date)"')

    t_python >> t_bash
'''

with open(f"{AIRFLOW_HOME}/dags/m2_mieszane_operatory.py", "w") as f:
    f.write(dag_kod)

airflow("dags", "test", "m2_mieszane_operatory", "2024-01-01")


<a id="sec5"></a>
## 5. XComs — przekazywanie danych między taskami

**XCom** (*cross-communication*) to mechanizm przekazywania MAŁYCH wartości między
taskami -- np. task A liczy coś i task B potrzebuje tego wyniku. Funkcja w
`PythonOperator` **zwraca** wartość (Airflow automatycznie zapisuje ją jako XCom), a
kolejny task ODCZYTUJE ją przez argument `ti` (*task instance*) w swojej funkcji.

In [ ]:
dag_kod = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime


def policz_liczbe_zamowien():
    liczba = 3000  # w prawdziwym DAG-u odczytana z bazy danych
    print(f"Policzono {liczba} zamowien.")
    return liczba  # Airflow automatycznie zapisuje to jako XCom


def wypisz_podsumowanie(ti):
    liczba = ti.xcom_pull(task_ids="policz_zamowien")
    print(f"Podsumowanie: w bazie jest {liczba} zamowien.")


with DAG(
    dag_id="m2_xcom_demo",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    t_policz = PythonOperator(task_id="policz_zamowien", python_callable=policz_liczbe_zamowien)
    t_podsumuj = PythonOperator(task_id="wypisz_podsumowanie", python_callable=wypisz_podsumowanie)

    t_policz >> t_podsumuj
'''

with open(f"{AIRFLOW_HOME}/dags/m2_xcom_demo.py", "w") as f:
    f.write(dag_kod)

airflow("dags", "test", "m2_xcom_demo", "2024-01-01")


> ⚠️ **XCom to NIE jest miejsce na duże dane**
>
> XCom-y są zapisywane w metadata bazie Airflow (u nas SQLite) -- świetne dla pojedynczych liczb, krótkich napisów czy małych słowników, ale NIGDY dla całych DataFrame'ów czy plików. Duże dane przekazuje się między krokami przez wspólny magazyn (plik, tabelę w hurtowni) -- task A zapisuje dane do Parquet, task B je odczytuje z tej samej ścieżki.

<a id="sec6"></a>
## 6. Harmonogramy: cron, presety, catchup

Parametr `schedule` DAG-a decyduje, kiedy scheduler ma go uruchamiać automatycznie (w
naszych przykładach używaliśmy `schedule=None`, czyli "tylko ręcznie"):

- **Wyrażenie cron**, np. `"0 6 * * *"` -- codziennie o 6:00 (dokładnie ta sama składnia
  co w Databricks Jobs, Moduł 14 poprzedniego kursu).
- **Presety**, np. `"@daily"` (codziennie o północy), `"@hourly"`, `"@weekly"` -- czytelne
  skróty dla najczęstszych przypadków.
- **`timedelta`**, np. `timedelta(hours=4)` -- co dokładnie 4 godziny od `start_date`.

**`catchup`** decyduje, co się dzieje, gdy DAG ma `start_date` w przeszłości: `True`
(domyślnie) każe Airflow uruchomić WSZYSTKIE zaległe wykonania od `start_date` do dziś
(np. jeśli DAG miał `start_date` sprzed miesiąca i harmonogram dzienny, uruchomi się 30
razy z rzędu!). `catchup=False` (używane w każdym przykładzie w tym module) uruchamia
tylko NAJNOWSZE zaplanowane wykonanie -- prawie zawsze to, czego chcesz przy nauce i
większości nowych DAG-ów.

> 🏭 **Dlaczego to ważne w praktyce**
>
> Zapomnienie o `catchup=False` na DAG-u ze `start_date` sprzed miesiąca to klasyczny błąd początkujących w Airflow -- system nagle próbuje uruchomić 30 zaległych wykonań naraz, co może przeciążyć bazę danych albo wysłać 30 duplikatów tego samego powiadomienia e-mail.

<a id="sec7"></a>
## 7. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- architekturę Airflow (Scheduler, Webserver, Executor, Metadata DB),
- DAG jako opis potoku i `PythonOperator` jako pierwszy typ zadania,
- zależności między taskami (`>>`, listy dla równoległości),
- `BashOperator` i mieszanie różnych typów operatorów w jednym DAG-u,
- XComs -- przekazywanie małych wartości między taskami,
- harmonogramy (cron, presety) i `catchup`.

### 📝 Ćwiczenie 1: DAG z trzema krokami sekwencyjnymi

Napisz DAG `m2_cw1_sekwencja` z trzema zadaniami `PythonOperator` (`krok_a`, `krok_b`, `krok_c`), z których każde po prostu wypisuje swoją nazwę, uruchamianymi ŚCIŚLE po kolei (a nie równolegle). Uruchom go przez `airflow dags test`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
dag_kod = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime


def krok(nazwa):
    def wykonaj():
        print(f"Wykonuje {nazwa}")
    return wykonaj


with DAG(
    dag_id="m2_cw1_sekwencja",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    a = PythonOperator(task_id="krok_a", python_callable=krok("krok_a"))
    b = PythonOperator(task_id="krok_b", python_callable=krok("krok_b"))
    c = PythonOperator(task_id="krok_c", python_callable=krok("krok_c"))

    a >> b >> c
'''
with open(f"{AIRFLOW_HOME}/dags/m2_cw1_sekwencja.py", "w") as f:
    f.write(dag_kod)

airflow("dags", "test", "m2_cw1_sekwencja", "2024-01-01")
```

</details>

### 📝 Ćwiczenie 2: Dwa niezależne zadania na starcie

Zmodyfikuj DAG z ćwiczenia 1 tak, żeby `krok_a` i `krok_b` uruchamiały się RÓWNOLEGLE na starcie, a `krok_c` czekał na oba, zanim się uruchomi.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
dag_kod = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime


def krok(nazwa):
    def wykonaj():
        print(f"Wykonuje {nazwa}")
    return wykonaj


with DAG(
    dag_id="m2_cw2_rownolegle",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    a = PythonOperator(task_id="krok_a", python_callable=krok("krok_a"))
    b = PythonOperator(task_id="krok_b", python_callable=krok("krok_b"))
    c = PythonOperator(task_id="krok_c", python_callable=krok("krok_c"))

    [a, b] >> c
'''
with open(f"{AIRFLOW_HOME}/dags/m2_cw2_rownolegle.py", "w") as f:
    f.write(dag_kod)

airflow("dags", "test", "m2_cw2_rownolegle", "2024-01-01")
```

</details>

### 📝 Ćwiczenie 3: Przekaż wynik przez XCom

Napisz DAG z dwoma zadaniami: pierwsze zwraca (przez `return`) listę trzech miast (`["Warszawa", "Krakow", "Gdansk"]`), drugie odczytuje ją przez XCom i wypisuje liczbę miast na liście.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
dag_kod = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime


def zwroc_miasta():
    return ["Warszawa", "Krakow", "Gdansk"]


def policz_miasta(ti):
    miasta = ti.xcom_pull(task_ids="zwroc_miasta")
    print(f"Liczba miast: {len(miasta)}")


with DAG(
    dag_id="m2_cw3_xcom",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    t1 = PythonOperator(task_id="zwroc_miasta", python_callable=zwroc_miasta)
    t2 = PythonOperator(task_id="policz_miasta", python_callable=policz_miasta)
    t1 >> t2
'''
with open(f"{AIRFLOW_HOME}/dags/m2_cw3_xcom.py", "w") as f:
    f.write(dag_kod)

airflow("dags", "test", "m2_cw3_xcom", "2024-01-01")
```

</details>

> 🔥 **Wyzwanie: mini-potok sklepu internetowego**
>
> Zbuduj DAG `m2_wyzwanie_sklep` z czterema zadaniami: `wczytaj_zamowienia` (zwraca przez XCom liczbę zamówień, np. 3000), `wczytaj_klientow` (zwraca liczbę klientów, np. 300) -- oba RÓWNOLEGLE na starcie -- a następnie `policz_sredni_wskaznik` (odczytuje OBIE wartości przez `ti.xcom_pull(task_ids=...)` i wypisuje średnią liczbę zamówień na klienta), i na końcu `wyslij_raport` (wypisuje "Raport wyslany").

<details>
<summary>Pokaż rozwiązanie</summary>

```python
dag_kod = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime


def wczytaj_zamowienia():
    return 3000


def wczytaj_klientow():
    return 300


def policz_sredni_wskaznik(ti):
    liczba_zamowien = ti.xcom_pull(task_ids="wczytaj_zamowienia")
    liczba_klientow = ti.xcom_pull(task_ids="wczytaj_klientow")
    wskaznik = liczba_zamowien / liczba_klientow
    print(f"Srednio {wskaznik:.2f} zamowien na klienta.")


def wyslij_raport():
    print("Raport wyslany.")


with DAG(
    dag_id="m2_wyzwanie_sklep",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
) as dag:
    t_zamowienia = PythonOperator(task_id="wczytaj_zamowienia", python_callable=wczytaj_zamowienia)
    t_klienci = PythonOperator(task_id="wczytaj_klientow", python_callable=wczytaj_klientow)
    t_wskaznik = PythonOperator(task_id="policz_sredni_wskaznik", python_callable=policz_sredni_wskaznik)
    t_raport = PythonOperator(task_id="wyslij_raport", python_callable=wyslij_raport)

    [t_zamowienia, t_klienci] >> t_wskaznik >> t_raport
'''
with open(f"{AIRFLOW_HOME}/dags/m2_wyzwanie_sklep.py", "w") as f:
    f.write(dag_kod)

airflow("dags", "test", "m2_wyzwanie_sklep", "2024-01-01")
```

To dokladnie ten sam wzorzec co w prawdziwych potokach: dwa niezalezne zrodla danych wczytywane rownolegle, polaczone w jednym kroku analitycznym, a na koncu akcja koncowa (raport/powiadomienie).

</details>

## Co dalej?

W **Module 3** poznamy zaawansowane wzorce Airflow: sensory (czekanie na warunek),
branching (wybór ścieżki w zależności od danych), dynamic task mapping oraz jak Airflow
wywołuje narzędzia zewnętrzne (dbt, Spark) w prawdziwych potokach danych.